In [1]:
import numpy as np
from glob import glob
from DNN.lw_spacenet import SeismicDataset, SeismicTrainer, create_data_loaders, UNet3D

In [2]:
def slice_data_chunks_with_stride(data, chunk_size=(128, 128, 128), stride=None):
    """
    Slice input data of shape (150, 150, 2000) into overlapping chunks of specified size.
    
    Args:
        data: Input numpy array of shape (150, 150, 2000)
        chunk_size: Tuple of (height, width, depth) for each chunk
        stride: Step size between chunks (default is half the chunk size for 50% overlap)
    
    Returns:
        List of numpy arrays, each of shape chunk_size
    """
    h, w, d = data.shape
    ch, cw, cd = chunk_size
    
    # Default stride is half the chunk size for 50% overlap
    if stride is None:
        stride = (ch // 2, cw // 2, cd // 2)
    
    sh, sw, sd = stride
    
    chunks = []
    
    # Calculate how many chunks we can fit in each dimension with the given stride
    h_steps = max(1, (h - ch) // sh + 1)
    w_steps = max(1, (w - cw) // sw + 1)
    d_steps = max(1, (d - cd) // sd + 1)
    
    for i in range(h_steps):
        for j in range(w_steps):
            for k in range(d_steps):
                start_h, end_h = i * sh, i * sh + ch
                start_w, end_w = j * sw, j * sw + cw
                start_d, end_d = k * sd, k * sd + cd
                
                # Ensure we don't exceed the data bounds
                end_h = min(end_h, h)
                end_w = min(end_w, w)
                end_d = min(end_d, d)
                
                # Adjust start positions if we're near the boundary
                start_h = max(0, end_h - ch)
                start_w = max(0, end_w - cw)
                start_d = max(0, end_d - cd)
                
                chunk = data[start_h:end_h, start_w:end_w, start_d:end_d]
                chunks.append(chunk)
    del chunks[0:3]
    return chunks

def robust_normalize_seismic(chunk, p_min=1, p_max=99):
    """Robust normalization using percentiles for seismic [-1, 1]"""
    q_min, q_max = np.percentile(chunk, [p_min, p_max])
    if q_max == q_min:
        return np.zeros_like(chunk)
    # Scale to [-1, 1] using robust percentiles
    normalized = 2 * (chunk - q_min) / (q_max - q_min) - 1
    # Optional: clip remaining outliers beyond the percentile range
    return np.clip(normalized, -1, 1)

def normalize_age_chunk(chunk):
    """Normalize age data to [0, 1] range within chunk"""
    min_val, max_val = chunk.min(), chunk.max()
    if max_val == min_val:  # Avoid division by zero
        return np.zeros_like(chunk)
    normalized = (chunk - min_val) / (max_val - min_val)
    return normalized

In [3]:
seismic = glob("../data/synthetic_data/run/*150-150-2000*/seismicCubes_cumsum_fullstack*.npy")
age = glob("../data/synthetic_data/run/*150-150-2000*/faulted_age*.npy")
seis_chunk_norm = []
age_chunk = []
for s, a in zip(seismic, age):
    sdf = np.load(s)
    sdf_norm = robust_normalize_seismic(sdf)
    adf = np.load(a)
    seis_chunk_norm.extend(slice_data_chunks_with_stride(sdf_norm))
    age_chunk.extend(slice_data_chunks_with_stride(adf))
age_chunk_norm = []
for chunk in age_chunk:
    age_chunk_norm.append(normalize_age_chunk(chunk))

In [ ]:
from util.plotting import plot_3d_array_interactive
plot_3d_array_interactive(age_chunk_norm[2], axis='x')
plot_3d_array_interactive(seis_chunk_norm[2], axis='x')

interactive(children=(IntSlider(value=0, description='X-index:', max=127), Output()), _dom_classes=('widget-in…

interactive(children=(IntSlider(value=0, description='X-index:', max=127), Output()), _dom_classes=('widget-in…

In [9]:
dataset = SeismicDataset(seis_chunk_norm, age_chunk_norm)
train_loader, val_loader = create_data_loaders(dataset, batch_size=3)
model = UNet3D(in_channels=1, out_channels=1, init_features=16)
trainer = SeismicTrainer(model, train_loader, val_loader)

/home/spaceswimmer/miniconda3/envs/diploma/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [10]:
trainer.train(num_epochs=20, save_dir="../data/DNN models")


Epoch 1/20
------------------------------
Batch 0/22, Loss: 0.632812
Batch 10/22, Loss: 0.632812
Batch 20/22, Loss: 0.566406
Train Loss: 0.592596
Val Loss: 0.501953
Saved best model with val_loss: 0.501953

Epoch 2/20
------------------------------
Batch 0/22, Loss: 0.458984
Batch 10/22, Loss: 0.474609
Batch 20/22, Loss: 0.339844
Train Loss: 0.435991
Val Loss: 0.348633
Saved best model with val_loss: 0.348633

Epoch 3/20
------------------------------
Batch 0/22, Loss: 0.359375
Batch 10/22, Loss: 0.343750
Batch 20/22, Loss: 0.242188
Train Loss: 0.306197
Val Loss: 0.256999
Saved best model with val_loss: 0.256999

Epoch 4/20
------------------------------
Batch 0/22, Loss: 0.200195
Batch 10/22, Loss: 0.247070
Batch 20/22, Loss: 0.229492
Train Loss: 0.229980
Val Loss: 0.201335
Saved best model with val_loss: 0.201335

Epoch 5/20
------------------------------
Batch 0/22, Loss: 0.200195
Batch 10/22, Loss: 0.190430
Batch 20/22, Loss: 0.176758
Train Loss: 0.194691
Val Loss: 0.548014

Epoch

([0.5925958806818182,
  0.43599076704545453,
  0.30619673295454547,
  0.22998046875,
  0.19469105113636365,
  0.18634588068181818,
  0.17418323863636365,
  0.16481711647727273,
  0.15653852982954544,
  0.15556196732954544,
  0.15032404119318182,
  0.14455344460227273,
  0.13896040482954544,
  0.13163618607954544,
  0.12830699573863635,
  0.12089399857954546,
  0.12000621448863637,
  0.11776455965909091,
  0.13285688920454544,
  0.120361328125],
 [0.501953125,
  0.3486328125,
  0.2569986979166667,
  0.20133463541666666,
  0.5480143229166666,
  0.23046875,
  0.18033854166666666,
  0.15462239583333334,
  0.22762044270833334,
  0.141357421875,
  0.18229166666666666,
  0.12459309895833333,
  0.11686197916666667,
  0.11946614583333333,
  0.1669921875,
  0.14558919270833334,
  0.13387044270833334,
  0.4735514322916667,
  0.121826171875,
  0.10302734375])